# EXP_060D — Alternative Combination 3: EfficientNet-B3 + ViSoBERT + Cross-Attention + Log-Cosh
**Phase 6 | Promising Combination Validation**
This notebook tests Alternative Candidate 3 from Phase 6.
- Image: EfficientNet-B3 | Text: ViSoBERT | Fusion: Cross-Attention | Loss: Log-Cosh | Seed: 42

### STEP 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### STEP 2: Clone source code and install dependencies

In [ ]:
!git clone https://github.com/lechihoang/SE365.git
%cd SE365
!pip install -r requirements.txt -q

Cloning into 'SE365'...
remote: Enumerating objects: 13467, done.
remote: Counting objects: 100% (338/338), done.
remote: Compressing objects: 100% (207/207), done.
remote: Total 13467 (delta 247), reused 219 (delta 131), pack-reused 13129 (from 1)
Receiving objects: 100% (13467/13467), 873.24 MiB | 32.72 MiB/s, done.
Resolving deltas: 100% (485/485), done.
/content/SE365


### STEP 3: Download and extract data

In [ ]:
!rm -rf ./data
!cp /content/drive/MyDrive/SE365/data.zip ./data.zip
!unzip -q data.zip
!rm data.zip
!ls -la ./data

total 1420
drwxr-xr-x  4 root root    4096 Jun 16 09:21 .
drwxr-xr-x 11 root root    4096 Jun 24 11:13 ..
drwxr-xr-x  2 root root 1437696 Jun 16 09:59 image
drwxr-xr-x  2 root root    4096 Jun 16 09:21 text


### STEP 4: Configure paths

In [ ]:
import os
DRIVE_ROOT = '/content/drive/MyDrive/SE365'  # ✏️ Change if needed
EXP_ID = 'EXP_060D_efficientnetb3_visobert_crossattention_logcosh'

BEST_IMAGE_EXP_ID = 'EXP_020D_efficientnetb3_xlmr_concat_mse'
BEST_TEXT_EXP_ID  = 'EXP_030D_bestimage_visobert_concat_mse'

DRIVE_EXP_PATH = f'{DRIVE_ROOT}/experiments/{EXP_ID}'
os.makedirs(DRIVE_EXP_PATH, exist_ok=True)
print(f'Artifacts: {DRIVE_EXP_PATH}')


Artifacts: /content/drive/MyDrive/SE365/experiments/EXP_060D_efficientnetb3_visobert_crossattention_logcosh


### STEP 5: Load pretrained weights

In [ ]:
import os, shutil
os.makedirs('./checkpoints', exist_ok=True)
shutil.copy(f'{DRIVE_ROOT}/experiments/{BEST_TEXT_EXP_ID}/best_model_train_text.pth', './checkpoints/best_model_train_text.pth')
print(f'Loaded text from {BEST_TEXT_EXP_ID}')
shutil.copy(f'{DRIVE_ROOT}/experiments/{BEST_IMAGE_EXP_ID}/best_model_train_image.pth', './checkpoints/best_model_train_image.pth')
print(f'Loaded image from {BEST_IMAGE_EXP_ID}')

Loaded text from EXP_030D_bestimage_visobert_concat_mse
Loaded image from EXP_020D_efficientnetb3_xlmr_concat_mse


### STEP 6: Train

In [ ]:
!python main.py \
  --mode train_fusion \
  --fusion_type cross_attention \
  --text_model_name uitnlp/visobert \
  --image_model_name efficientnet_b3 \
  --loss_fn logcosh \
  --epochs 15 \
  --batch_size 16 \
  --lr 1e-5 \
  --grad_accum_steps 2 \
  --patience 5 \
  --unfreeze_text_layers 1 \
  --unfreeze_image_layers 1 \
  --seed 42 \
  --use_amp \
  --exp_id EXP_060D_efficientnetb3_visobert_crossattention_logcosh \
  --exp_dir ./experiments


====== MODE: TRAIN_FUSION ======
Using device: cuda
Seed: 42 | Experiment: EXP_060D_efficientnetb3_visobert_crossattention_logcosh
config.json: 100% 644/644 [00:00<00:00, 3.74MB/s]
sentencepiece.bpe.model: 100% 471k/471k [00:01<00:00, 360kB/s]
Loaded timm processor for efficientnet_b3
pytorch_model.bin: 100% 390M/390M [00:08<00:00, 48.6MB/s]
Loading weights: 100% 197/197 [00:00<00:00, 15708.41it/s]
[transformers] XLMRobertaModel LOAD REPORT from: uitnlp/visobert
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly

### STEP 7: Evaluate on Test Set
Evaluate the best model on the unseen test set to report final metrics and generate plots.

In [ ]:
!python test.py \
  --mode train_fusion \
  --fusion_type cross_attention \
  --text_model_name uitnlp/visobert \
  --image_model_name efficientnet_b3 \
  --loss_fn logcosh \
  --exp_id EXP_060D_efficientnetb3_visobert_crossattention_logcosh \
  --exp_dir ./experiments


====== TESTING: TRAIN_FUSION ======
Device: cuda
Test samples: 600
Loading weights: 100% 197/197 [00:00<00:00, 14580.26it/s]
[transformers] XLMRobertaModel LOAD REPORT from: uitnlp/visobert
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loaded weights: ./checkpoints/best_model_train_fusion.pth
/content/SE365/test.py:114: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('c

### STEP 8: Save to Drive + print metrics


In [ ]:
import json
!cp -r ./experiments/$EXP_ID/* $DRIVE_EXP_PATH/

# --- VALIDATION METRICS ---
with open(f'./experiments/{EXP_ID}/metrics.json') as f:
    m = json.load(f)

print(f'\n=== {EXP_ID} Results (Validation) ===')
print(f"Loss (val)   : {m['loss']:.4f}")
print()
print("             MAE      RMSE      R2")
print(f"  food     : {m['mae_food']:.4f}   {m['rmse_food']:.4f}   {m['r2_food']:.4f}")
print(f"  price    : {m['mae_price']:.4f}   {m['rmse_price']:.4f}   {m['r2_price']:.4f}")
print(f"  atmos    : {m['mae_atmos']:.4f}   {m['rmse_atmos']:.4f}   {m['r2_atmos']:.4f}")
print(f"  service  : {m['mae_service']:.4f}   {m['rmse_service']:.4f}   {m['r2_service']:.4f}")
print(f"  overall  : {m['mae_overall']:.4f}   {m['rmse_overall']:.4f}   {m['r2_overall']:.4f}")
print()
print(f"  mean_mae   : {m['mean_mae']:.4f}")
print(f"  aspect_mae : {m['aspect_mae']:.4f}")
print(f"  overall_mae: {m['overall_mae']:.4f}")

# --- TEST METRICS ---
with open(f'./experiments/{EXP_ID}/test_metrics.json') as f:
    t = json.load(f)

print(f'\n=== {EXP_ID} Results (Test) ===')
print()
print("             MAE      RMSE      R2")
print(f"  food     : {t['mae_food']:.4f}   {t['rmse_food']:.4f}   {t['r2_food']:.4f}")
print(f"  price    : {t['mae_price']:.4f}   {t['rmse_price']:.4f}   {t['r2_price']:.4f}")
print(f"  atmos    : {t['mae_atmos']:.4f}   {t['rmse_atmos']:.4f}   {t['r2_atmos']:.4f}")
print(f"  service  : {t['mae_service']:.4f}   {t['rmse_service']:.4f}   {t['r2_service']:.4f}")
print(f"  overall  : {t['mae_overall']:.4f}   {t['rmse_overall']:.4f}   {t['r2_overall']:.4f}")
print()
print(f"  mean_mae   : {t['mean_mae']:.4f}")
print(f"  aspect_mae : {t['aspect_mae']:.4f}")
print(f"  overall_mae: {t['overall_mae']:.4f}")



=== EXP_060D_efficientnetb3_visobert_crossattention_logcosh Results (Validation) ===
Loss (val)   : 0.7925

             MAE      RMSE      R2
  food     : 1.3120   1.7891   0.3918
  price    : 1.3630   1.8138   0.2634
  atmos    : 1.2749   1.6758   0.2764
  service  : 1.3296   1.7647   0.3926
  overall  : 1.1353   1.5289   0.4259

  mean_mae   : 1.2829
  aspect_mae : 1.3199
  overall_mae: 1.1353

=== EXP_060D_efficientnetb3_visobert_crossattention_logcosh Results (Test) ===

             MAE      RMSE      R2
  food     : 1.2764   1.7052   0.4738
  price    : 1.2672   1.6440   0.3560
  atmos    : 1.3083   1.6600   0.2719
  service  : 1.2477   1.6629   0.4170
  overall  : 1.0787   1.4067   0.4972

  mean_mae   : 1.2357
  aspect_mae : 1.2749
  overall_mae: 1.0787
